# 04 — Recorte espacial e viabilidade de dados externos

Este notebook mede as duas coisas que `docs/04-dados-externos.md` deixou
condicionadas, e nada mais. Não modela.

**1. Expandir o recorte vale a pena?** D-16 coloca isso como a primeira coisa a
fazer, antes de qualquer fonte externa: é mais barato — o dado já está baixado e o
filtro é um parâmetro — e não introduz fonte de erro nova. O que se ganha é
variância territorial, e com ela a possibilidade de usar população municipal do
IBGE, que existe anualmente por município e dispensa mediação geográfica.

**2. Quem fica de fora da trilha geográfica?** D-17 obriga caracterizar os não
posicionáveis. Se eles diferirem sistematicamente em tipo ou gestão, a trilha
geográfica não fala sobre a rede, fala sobre um recorte enviesado dela.

**Estado.** As duas perguntas estão fechadas: a primeira em D-21, que expandiu o
recorte para o estado; a segunda em D-41, escrita a partir deste notebook. As
células continuam aqui como registro de procedência e para remedir quando a série
crescer.

In [1]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import duckdb
import pandas as pd

from src.config.paths import PRIMARY_FOLDER
from src.etl import changes

pd.set_option("display.width", 200)
con = duckdb.connect()
PERIODOS = changes.periodos_disponiveis()
ULTIMO = PERIODOS[-1]
SP = "355030"

raiz = lambda p: PRIMARY_FOLDER / p / "tbEstabelecimento.parquet"
print(f"snapshots: {PERIODOS}\nreferência para os cortes: {ULTIMO}")

snapshots: ['201701', '201801', '201901', '202001', '202101', '202201', '202301', '202401', '202501', '202601']
referência para os cortes: 202601


## 1. Quanto cada recorte acrescenta

Três recortes possíveis, do mais estreito ao mais largo. O que interessa não é só
o número de nós: é quantos **municípios distintos** entram, porque é isso que dá
variância à população do IBGE.

A Região Metropolitana de São Paulo tem 39 municípios; os códigos IBGE de todos
começam com `35`, como todo o estado, então o recorte da RM precisa de lista
explícita.

In [2]:
# Os 39 municípios da RMSP, códigos IBGE de 7 dígitos.
RMSP = [
    "3503901", "3505708", "3506607", "3509007", "3509205", "3510609", "3513009",
    "3513801", "3515004", "3515103", "3515707", "3516309", "3516408", "3518800",
    "3522208", "3522505", "3523107", "3525003", "3526209", "3528502", "3529401",
    "3530607", "3534401", "3539806", "3543303", "3544103", "3545001", "3546801",
    "3547304", "3547809", "3548708", "3548807", "3549904", "3550308", "3552502",
    "3552809", "3556453", "3557105", "3505500",
]
# O CNES grava co_municipio_gestor com 6 dígitos (sem o verificador).
RMSP6 = sorted({m[:6] for m in RMSP})

def perfil_recorte(nome: str, filtro: str) -> dict:
    r = con.execute(f'''
        SELECT COUNT(DISTINCT co_unidade) estabelecimentos,
               COUNT(DISTINCT co_municipio_gestor) municipios,
               COUNT(DISTINCT CASE WHEN nu_latitude IS NOT NULL
                                    AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                                   THEN co_unidade END) com_coordenada
        FROM read_parquet('{raiz(ULTIMO)}') WHERE {filtro}
    ''').fetchone()
    return {"recorte": nome, "estabelecimentos": r[0], "municipios": r[1],
            "com_coordenada": r[2],
            "cobertura_%": round(100 * r[2] / r[0], 1) if r[0] else 0.0}

lista_rm = ", ".join(f"'{m}'" for m in RMSP6)
recortes = pd.DataFrame([
    perfil_recorte("município de SP", f"co_municipio_gestor = '{SP}'"),
    perfil_recorte("RMSP (39 municípios)", f"co_municipio_gestor IN ({lista_rm})"),
    perfil_recorte("estado de SP", "co_municipio_gestor LIKE '35%'"),
    perfil_recorte("Brasil", "1 = 1"),
])
print(recortes.to_string(index=False))

             recorte  estabelecimentos  municipios  com_coordenada  cobertura_%
     município de SP             42027           1           32846         78.2
RMSP (39 municípios)             61212          39           49655         81.1
        estado de SP            146500         645          127859         87.3
              Brasil            602160        5585          542462         90.1


### O que o recorte custa em tempo de execução

Expandir não é grátis: o grafo relacional cresce com o número de nós, e o espaço
de candidatos da tarefa de aquisição cresce com estabelecimentos × tipos de
equipamento. Medindo o tamanho do espaço de rótulos em cada recorte, para que a
decisão seja informada e não otimista.

In [3]:
def tamanho_da_tarefa(nome: str, filtro: str) -> dict:
    equip = PRIMARY_FOLDER / ULTIMO / "rlEstabEquipamento.parquet"
    r = con.execute(f'''
        WITH sel AS (
            SELECT DISTINCT co_unidade FROM read_parquet('{raiz(ULTIMO)}')
            WHERE {filtro}
        ),
        itens AS (
            SELECT DISTINCT co_equipamento FROM read_parquet('{equip}')
            WHERE co_equipamento IS NOT NULL
        ),
        tem AS (
            SELECT DISTINCT co_unidade, co_equipamento
            FROM read_parquet('{equip}') JOIN sel USING (co_unidade)
        )
        SELECT (SELECT COUNT(*) FROM sel) AS estab,
               (SELECT COUNT(*) FROM itens) AS itens,
               (SELECT COUNT(*) FROM tem) AS pares_existentes
    ''').fetchone()
    return {"recorte": nome, "estabelecimentos": r[0], "tipos_equipamento": r[1],
            "pares_existentes": r[2],
            "candidatos_por_transicao": r[0] * r[1] - r[2]}

tamanhos = pd.DataFrame([
    tamanho_da_tarefa("municipio de SP", f"co_municipio_gestor = '{SP}'"),
    tamanho_da_tarefa("RMSP", f"co_municipio_gestor IN ({lista_rm})"),
    tamanho_da_tarefa("estado de SP", "co_municipio_gestor LIKE '35%'"),
])

# O numero de transicoes vem da serie, nunca fixo: cada competencia nova
# acrescenta uma, e um literal aqui produziria um total errado em silencio.
N_TRANSICOES = len(PERIODOS) - 1
tamanhos[f"candidatos_{N_TRANSICOES}_transicoes"] = (
    tamanhos["candidatos_por_transicao"] * N_TRANSICOES
)
print(tamanhos.to_string(index=False))
print(f"\nO espaco de candidatos e o custo dominante: multiplica por {N_TRANSICOES}")
print("transicoes, e cada linha vira um exemplo de treino.")

        recorte  estabelecimentos  tipos_equipamento  pares_existentes  candidatos_por_transicao  candidatos_9_transicoes
municipio de SP             42027                 99             91430                   4069243                 36623187
           RMSP             61212                 99            142624                   5917364                 53256276
   estado de SP            146500                 99            326140                  14177360                127596240

O espaco de candidatos e o custo dominante: multiplica por 9
transicoes, e cada linha vira um exemplo de treino.


> **Veredito — recorte espacial: o estado.** Registrado em D-21.
>
> A RMSP acrescenta 39 municípios por 61,2 mil estabelecimentos; o estado
> acrescenta **645 municípios** por 146,5 mil. Como o objetivo do recorte mais largo
> era dar variância à população do IBGE, 39 municípios seriam pouco e o custo em
> candidatos é da mesma ordem. `RECORTE_PADRAO` passou a ser `'35'`, e o recorte
> deixou de ser código de município para ser **prefixo de código IBGE**: `'355030'`
> devolve a capital, `None` o país.
>
> Ganho medido: 2,9× mais eventos de aquisição, e cobertura de coordenada de 87,3%
> contra 78,2%. Perda: prevalência cai de 0,065% para 0,047%, porque o estado tem
> proporcionalmente mais estabelecimentos pequenos.

## 2. Quem fica de fora da trilha geográfica

Se os não posicionáveis se distribuírem como os posicionáveis, a restrição custa
poder estatístico mas não introduz viés. Se não, a trilha 3 fala sobre um
subconjunto e isso precisa constar no reporte.

A medição roda sobre o **recorte estadual**, que é o vigente desde D-21. Medir na
capital — como este notebook fazia — infla o viés: lá a cobertura é 78,2% contra
87,3% no estado, e a capital concentra consultório de pessoa física, justamente a
categoria mal geocodificada.

In [4]:
ATRIBUTOS = ["tp_unidade", "tp_gestao", "nivel_dep", "tp_pfpj", "co_natureza_jur"]

# O recorte vigente e o estado (D-21), nao a capital. Medir na capital dava um
# vies inflado: 78,2% de cobertura contra 87,3% no estado (D-41).
FILTRO_RECORTE = "co_municipio_gestor LIKE '35%'"

def comparar_posicionaveis(filtro: str, atributo: str) -> pd.DataFrame:
    return con.execute(f'''
        WITH e AS (
            SELECT "{atributo}" AS valor,
                   (nu_latitude IS NOT NULL
                    AND NOT (nu_latitude = 0 AND nu_longitude = 0)) AS posicionavel
            FROM read_parquet('{raiz(ULTIMO)}') WHERE {filtro}
        )
        SELECT valor,
               SUM(CASE WHEN posicionavel THEN 1 ELSE 0 END) AS posicionavel,
               SUM(CASE WHEN posicionavel THEN 0 ELSE 1 END) AS sem_coordenada,
               COUNT(*) AS total
        FROM e GROUP BY valor ORDER BY total DESC
    ''').df()

for atributo in ATRIBUTOS:
    df = comparar_posicionaveis(FILTRO_RECORTE, atributo)
    if df.empty:
        continue
    df["cobertura_%"] = (100 * df["posicionavel"] / df["total"]).round(1)
    base = 100 * df["posicionavel"].sum() / df["total"].sum()
    df["desvio_pp"] = (df["cobertura_%"] - base).round(1)
    print(f"\n{'=' * 72}\n{atributo}  (cobertura media = {base:.1f}%)\n{'=' * 72}")
    print(df.head(12).to_string(index=False))


tp_unidade  (cobertura media = 87.3%)
valor  posicionavel  sem_coordenada  total  cobertura_%  desvio_pp
   22       81554.0         13143.0  94697         86.1       -1.2
   36       19157.0          2715.0  21872         87.6        0.3
   39        6764.0          1128.0   7892         85.7       -1.6
   02        5488.0           307.0   5795         94.7        7.4
   04        3884.0           316.0   4200         92.5        5.2
   43        2764.0            99.0   2863         96.5        9.2
   42        1146.0           179.0   1325         86.5       -0.8
   05         939.0           108.0   1047         89.7        2.4
   68         817.0            19.0    836         97.7       10.4
   70         647.0            16.0    663         97.6       10.3
   50         468.0           178.0    646         72.4      -14.9
   77         574.0            44.0    618         92.9        5.6

tp_gestao  (cobertura media = 87.3%)
valor  posicionavel  sem_coordenada  total  cobertur


nivel_dep  (cobertura media = 87.3%)
valor  posicionavel  sem_coordenada  total  cobertura_%  desvio_pp
    1      114549.0         16827.0 131376         87.2       -0.1
    3       13310.0          1814.0  15124         88.0        0.7

tp_pfpj  (cobertura media = 87.3%)
valor  posicionavel  sem_coordenada  total  cobertura_%  desvio_pp
    3       85977.0          8429.0  94406         91.1        3.8
    1       41882.0         10212.0  52094         80.4       -6.9

co_natureza_jur  (cobertura media = 87.3%)
valor  posicionavel  sem_coordenada  total  cobertura_%  desvio_pp
 4000       40939.0         10211.0  51150         80.0       -7.3
 2062       45383.0          2226.0  47609         95.3        8.0
 2240       11316.0          2342.0  13658         82.9       -4.4
 1244       11097.0           334.0  11431         97.1        9.8
 2135        6981.0           428.0   7409         94.2        6.9
 2232        3240.0           330.0   3570         90.8        3.5
 3999      

In [5]:
# Teste formal: a cobertura depende do atributo, ou e uniforme?
#
# O chi2 responde "existe dependencia?" e com dezenas de milhares de linhas da
# significativo para quase tudo — sozinho nao informa. O V de Cramer responde
# "quao forte?", numa escala de 0 a 1, e e o numero que se le.
from scipy.stats import chi2_contingency

print(f"{'atributo':22} {'chi2':>12} {'p':>12} {'V de Cramer':>12}")
for atributo in ATRIBUTOS:
    df = comparar_posicionaveis(FILTRO_RECORTE, atributo)
    df = df[(df["total"] >= 20) & df["valor"].notna()]
    if len(df) < 2:
        continue
    tabela = df[["posicionavel", "sem_coordenada"]].to_numpy()
    chi2, p, _, _ = chi2_contingency(tabela)
    n = tabela.sum()
    cramer = (chi2 / (n * (min(tabela.shape) - 1))) ** 0.5
    print(f"{atributo:22} {chi2:>12.1f} {p:>12.2e} {cramer:>12.3f}")

print("\nV perto de 0 = cobertura uniforme, exclusao so custa poder.")
print("V alto = a exclusao e seletiva e a trilha 3 fala de um subconjunto enviesado.")

atributo                       chi2            p  V de Cramer
tp_unidade                   1317.4    1.97e-257        0.095
tp_gestao                     935.3    1.92e-202        0.080
nivel_dep                       8.0     4.62e-03        0.007
tp_pfpj                      3443.5     0.00e+00        0.153
co_natureza_jur             17798.4     0.00e+00        0.349

V perto de 0 = cobertura uniforme, exclusao so custa poder.
V alto = a exclusao e seletiva e a trilha 3 fala de um subconjunto enviesado.


> **Veredito — viés de posicionabilidade: seletivo, mas não alcança os rótulos.**
> Registrado em D-41.
>
> A exclusão **não** é uniforme. No estado, V de Cramér de **0,349** em
> `co_natureza_jur` e **0,153** em `tp_pfpj`; `tp_unidade`, `tp_gestao` e
> `nivel_dep` ficam abaixo de 0,10. Pessoa jurídica é 91,1% coberta contra 80,4% da
> pessoa física — 10,7 pontos. Na capital a mesma diferença era de 26,6 pontos, o
> que mostra que o recorte estadual atenua o viés pela metade.
>
> Mas o viés **não chega ao rótulo**: os não posicionáveis são 12,7% dos
> estabelecimentos e 3,9% das aquisições, e nas duas transições mais recentes são
> **zero**. A comparação pareada custa pouco poder estatístico.
>
> Consequência no reporte: onde o texto disser que o subconjunto é "não aleatório",
> passa a dizer em quê — sobre-representa pessoa jurídica. Nenhuma mudança de código.

## 3. Pré-requisito da população do IBGE

D-16 libera a população municipal **se** o recorte expandir. O que falta verificar
é trivial mas não pode ser suposto: o código de município do CNES casa com o
código do IBGE?

O CNES grava `co_municipio_gestor` com 6 dígitos; o IBGE usa 7, sendo o último um
dígito verificador. A junção é determinística — truncar o código do IBGE — mas a
cobertura precisa ser medida, não assumida.

In [6]:
r = con.execute(f'''
    SELECT COUNT(DISTINCT co_municipio_gestor) municipios,
           SUM(CASE WHEN LENGTH(co_municipio_gestor) = 6 THEN 1 ELSE 0 END) com_6,
           SUM(CASE WHEN LENGTH(co_municipio_gestor) <> 6 THEN 1 ELSE 0 END) outro,
           COUNT(*) linhas
    FROM read_parquet('{raiz(ULTIMO)}') WHERE co_municipio_gestor LIKE '35%'
''').df()
print("Estado de SP, formato de co_municipio_gestor:")
print(r.to_string(index=False))

# O IBGE tem 645 municípios em SP. Quantos aparecem no CNES?
n = int(r["municipios"].iloc[0])
print(f"\nmunicípios distintos no CNES: {n} (IBGE registra 645 em SP)")
print("Se a contagem bater, a junção por truncamento do código IBGE é direta;")
print("qualquer divergência precisa ser listada antes de a população entrar.")

Estado de SP, formato de co_municipio_gestor:
 municipios    com_6  outro  linhas
        645 146500.0    0.0  146500

municípios distintos no CNES: 645 (IBGE registra 645 em SP)
Se a contagem bater, a junção por truncamento do código IBGE é direta;
qualquer divergência precisa ser listada antes de a população entrar.


> **Veredito — chave do IBGE: casa perfeitamente.**
>
> 645 municípios distintos no CNES contra os 645 que o IBGE registra em São Paulo.
> Todas as 146,5 mil linhas têm `co_municipio_gestor` com 6 dígitos, zero
> divergência. A junção por truncamento do código IBGE de 7 dígitos é determinística
> e não precisa de mediação geográfica: C1 e C2 de `04-dados-externos.md` passam.
>
> **O pré-requisito passa e a fonte mesmo assim não entra** — ver D-40. O bloqueio
> não é técnico: é que nenhum dos dois papéis previstos tem consumidor. Como
> atributo, contamina a trilha 1, que precisa ficar livre de informação territorial.
> Como denominador, não há onde o número ser lido — AP, AUC e MAP@k são
> adimensionais, e não existe seção descritiva nem notebook de explicabilidade.

## Decisão

| Item | Veredito | Onde ficou registrado |
|---|---|---|
| Recorte espacial | estado de São Paulo, prefixo `'35'` | D-21 |
| Viés de posicionabilidade | seletivo em natureza jurídica, não alcança os rótulos | D-41 |
| Chave do IBGE | casa, 645 de 645 | D-40 (adiada por outro motivo) |

O que **não** se decide aqui: SIA/SUS. Aquele teste exige baixar um mês de
produção ambulatorial e medir pareamento por `co_cnes` — trabalho próprio, com
dependência nova, descrito na seção 3.2 de `docs/04-dados-externos.md`.

### O que continua em aberto

- **Remedir quando a série crescer.** A cobertura de coordenada sobe cerca de 1,6
  ponto por ano, então o viés de D-41 tende a atenuar sozinho. As células acima
  refazem a medição sem edição.
- **Decompor `co_natureza_jur`.** O V de 0,349 pode estar dominado por categorias
  minúsculas — natureza 2000 tem 131 estabelecimentos e 0% de cobertura. Olhar
  categoria por categoria diria se o efeito tem peso prático.
- **A caixa de plausibilidade não entra nestas contas.** Aqui o critério é apenas
  coordenada não nula e diferente de zero; `src/ml/graph.py` aplica também um filtro
  geográfico. A diferença é de cerca de um ponto e não muda conclusão, mas os
  números não são intercambiáveis com os de D-22.